# END Engine — Colab run 5: four local fixes

Same as the smoke test, but it applies four Engine fixes that are not on GitHub yet: the verifier no longer sees earlier scores; the overclaim check no longer blocks definitions but catches invented track records; a revision that returns its parent unchanged stops the run (at the no-progress limit) instead of being judged again; and exploring branches no longer resets the no-progress count when the answer did not change. Defaults are set for this run: `qwen3:14b`, 5 cycles, 16,384-token context. Use an **L4** (a T4 also works, slower). Expect about 10–40 minutes depending on the GPU.

**What it does**
1. Checks that a GPU is attached.
2. Clones the END Engine from GitHub and runs the offline unit tests.
3. Installs Ollama, pulls `qwen3:4b`, and confirms the model is loaded on the GPU.
4. Runs one task through the **FULL_END_NATIVE** setup from the v0.3.5.1 pilot (same model settings, `max_cycles=3`).
5. Optionally runs a one-call **DIRECT** answer on the same task for comparison.
6. Saves the answers, trajectory, call log, GPU readings and logs as a zip (download or Google Drive).

**How to use**
1. **Runtime → Change runtime type → T4 GPU → Save**, then **Connect**.
2. Change the settings in section 1 if you want (task, model, Drive saving).
3. **Runtime → Run all**. Expect the full END run to take several minutes.

This is a smoke test on real hardware. It is not one of the preregistered Study A/B experiments, and Colab is a disposable worker: the
Ledger and Windows/NTFS evidence stay on your own machine. Colab deletes its disk when the session ends, so run section 9 before you leave.

## 1. Settings

In [ ]:
# --- Edit these if you want -------------------------------------------------
REPO_URL = "https://github.com/kaamikazi/The-End-Engine-.git"
BRANCH = "checkpoint/end-engine-v0352-research-20261013"
MODEL = "qwen3:14b"           # about 9 GB; use an L4 (a T4 also fits, slower)
MAX_CYCLES = 5
NUM_CTX = 16384               # 8192 cut 6 prompts in run 2
RUN_OFFLINE_TESTS = True      # the repo's own unit tests, no model needed
RUN_DIRECT_BASELINE = True    # one plain model call on the same task, for comparison
SAVE_TO_DRIVE = False         # True = copy the results zip to Google Drive (asks for permission)
DRIVE_FOLDER = "END_Engine_Colab_Runs"
ALLOW_CPU = False             # True = continue without a GPU (very slow)
APPLY_LOCAL_FIXES = True      # False = run the GitHub code unchanged (for comparison)
PATCH_B64 = "..."  # fixes made against commit d9c518756f; do not edit

TASK = (
    "Design a memory architecture for a long-running AI agent that must preserve useful "
    "information without allowing memory growth to become unbounded."
)
# -----------------------------------------------------------------------------

import datetime, os, pathlib

WORK = pathlib.Path("/content")
REPO = WORK / "end_engine_repo"
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = WORK / "end_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY = {"run_id": RUN_ID, "model": MODEL, "branch": BRANCH, "max_cycles": MAX_CYCLES, "num_ctx": NUM_CTX, "task": TASK}
print("Results folder:", RUN_DIR)

## 2. GPU check

In [ ]:
import shutil, subprocess

def sh(cmd, check=True, **kw):
    """Run a shell command, print its output and return it."""
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    out = (p.stdout or "") + (p.stderr or "")
    if out.strip():
        print(out.rstrip())
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return out

if shutil.which("nvidia-smi"):
    gpu = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False).strip()
    sh("nvidia-smi > " + str(RUN_DIR / "nvidia_smi_start.txt"), check=False)
else:
    gpu = ""
SUMMARY["gpu"] = gpu or None
if gpu:
    print("GPU found:", gpu)
elif ALLOW_CPU:
    print("No GPU. Continuing on CPU because ALLOW_CPU = True. This will be slow.")
else:
    raise SystemExit("No GPU attached. Use Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all again. "
                     "(Or set ALLOW_CPU = True in section 1.)")

## 3. Get the END Engine code

In [ ]:
import json, sys

if not (REPO / ".git").exists():
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}")
else:
    print("Repo already cloned; reusing it.")
COMMIT = sh(f"git -C {REPO} rev-parse HEAD").strip()
SUMMARY["commit"] = COMMIT

# --- Apply the local Engine fixes (not on GitHub yet) before anything imports end_engine -----------------
import base64, hashlib, zlib
PATCH = json.loads(zlib.decompress(base64.b64decode(PATCH_B64)))
if APPLY_LOCAL_FIXES:
    def file_hash(path):
        return hashlib.sha256((REPO / path).read_bytes()).hexdigest() if (REPO / path).exists() else None
    pending = [p for p, i in PATCH["files"].items() if file_hash(p) != i["sha256"]]
    if pending and "end_engine" in sys.modules:
        raise SystemExit("end_engine was already imported unpatched in this session. "
                         "Use Runtime -> Restart session, then Run all.")
    for path in pending:
        item, target, current = PATCH["files"][path], REPO / path, file_hash(path)
        if current != item["base_sha256"]:
            raise SystemExit(f"{path} in the clone is not the version these fixes were made against "
                             f"(base commit {PATCH['base_commit'][:10]}). Not applying anything.")
        target.write_bytes(base64.b64decode(item["content"]))
        print("patched", path)
    SUMMARY["local_fixes"] = {"base_commit": PATCH["base_commit"],
                              "files": {p: i["sha256"] for p, i in PATCH["files"].items()}}
else:
    SUMMARY["local_fixes"] = None
    print("Local fixes NOT applied (APPLY_LOCAL_FIXES = False).")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Python", sys.version.split()[0])
if sys.version_info < (3, 11):
    raise SystemExit("END Engine needs Python 3.11 or newer.")

import end_engine
print("END Engine imported from", pathlib.Path(end_engine.__file__).parent)

## 4. Offline unit tests

These use the engine's mock model, so they test the code, not the GPU. A few may fail on Linux if they depend on Windows file behaviour; the run continues either way.

In [ ]:
if RUN_OFFLINE_TESTS:
    p = subprocess.run([sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-t", "."],
                       cwd=REPO, text=True, capture_output=True)
    (RUN_DIR / "offline_tests.txt").write_text(p.stdout + p.stderr)
    tail = (p.stdout + p.stderr).strip().splitlines()[-4:]
    print("\n".join(tail))
    SUMMARY["offline_tests"] = {"passed": p.returncode == 0, "tail": tail}
    print("\nOffline tests:", "PASS" if p.returncode == 0 else "SOME FAILURES (see offline_tests.txt in the results)")
else:
    SUMMARY["offline_tests"] = "skipped"

## 5. Install Ollama and load the model on the GPU

In [ ]:
import json, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2) as r:
            return json.loads(r.read())["version"]
    except Exception:
        return None

if not shutil.which("ollama"):
    sh("apt-get -qq update && apt-get -qq install -y pciutils zstd > /dev/null", check=False)
    sh("curl -fsSL https://ollama.com/install.sh | sh > /content/ollama_install.log 2>&1")
    print("Ollama installed.")

if not ollama_up():
    OLLAMA_LOG = open(RUN_DIR / "ollama_serve.log", "a")
    subprocess.Popen(["ollama", "serve"], stdout=OLLAMA_LOG, stderr=subprocess.STDOUT,
                     env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
OLLAMA_VERSION = ollama_up()
if not OLLAMA_VERSION:
    raise SystemExit("Ollama server did not start. Check ollama_serve.log in the results folder.")
print("Ollama server version:", OLLAMA_VERSION)
SUMMARY["ollama_version"] = OLLAMA_VERSION

print(f"Pulling {MODEL} (first time takes a minute or two)...")
sh(f"ollama pull {MODEL} 2>&1 | tail -n 2")

# Warm-up call: loads the model into VRAM so we can confirm where it runs.
req = urllib.request.Request("http://127.0.0.1:11434/api/generate", headers={"Content-Type": "application/json"},
    data=json.dumps({"model": MODEL, "prompt": "Say OK.", "stream": False, "think": False,
                     "options": {"num_ctx": NUM_CTX, "num_predict": 8}}).encode())
with urllib.request.urlopen(req, timeout=600) as r:
    warm = json.loads(r.read())
print("Warm-up reply:", repr(warm.get("response", "")[:40]))
ps = sh("ollama ps", check=False)
SUMMARY["ollama_ps"] = ps.strip()
if gpu and "GPU" not in ps:
    print("WARNING: Ollama does not report the model on the GPU. The run will work but be slow. See ollama_serve.log.")

## 6. Run the END Engine (FULL_END_NATIVE, pilot settings)

In [ ]:
import json, subprocess, threading, time, traceback
from v0351_core_regression_pilot.bench.providers import CallLog, ModelSettings, OllamaModel, discover
from end_engine import EndEngine, EngineConfig
from end_engine.quality.providers import ModelJudgeProvider

class GpuSampler:
    """Polls nvidia-smi every 2 s while a run is going (peak memory, mean utilisation) and prints progress every 30 s."""
    def __init__(self, log):
        self.log, self.samples, self._stop = log, [], threading.Event()
    def _loop(self):
        t0, ticks = time.time(), 0
        while not self._stop.is_set():
            if gpu:
                try:
                    line = subprocess.run("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits",
                                          shell=True, text=True, capture_output=True, timeout=10).stdout.strip().splitlines()[0]
                    mem, util = (int(x.strip()) for x in line.split(","))
                    self.samples.append((mem, util))
                except Exception:
                    pass
            ticks += 1
            if ticks % 15 == 0:
                last = self.samples[-1] if self.samples else None
                print(f"  ... {int(time.time() - t0)} s, {len(self.log.entries)} model calls so far"
                      + (f", GPU {last[0]} MiB / {last[1]}%" if last else ""), flush=True)
            self._stop.wait(2)
    def __enter__(self):
        self._t = threading.Thread(target=self._loop, daemon=True); self._t.start()
        return self
    def __exit__(self, *exc):
        self._stop.set()
    def report(self):
        if not self.samples:
            return None
        return {"peak_memory_mib": max(m for m, _ in self.samples),
                "mean_utilization_pct": round(sum(u for _, u in self.samples) / len(self.samples), 1),
                "samples": len(self.samples)}

settings = ModelSettings(model=MODEL, num_ctx=NUM_CTX)
identity = discover(settings)            # model name + digest from the local Ollama server
SUMMARY["model_identity"] = identity
end_log = CallLog()
model = OllamaModel(settings, identity, end_log)
config = EngineConfig(ledger_dir=str(RUN_DIR / "ledger"), candidate_generation="native", protocol_repair_attempts=1)
engine = EndEngine(config, provider=model, judge_provider=ModelJudgeProvider(model))

print(f"Running END on {MODEL} (max_cycles={MAX_CYCLES}). It makes many model calls; progress prints every 30 s.")
end_result, end_error = None, None
start = time.time()
with GpuSampler(end_log) as end_gpu:
    try:
        end_result = engine.run(TASK, max_cycles=MAX_CYCLES)
    except Exception as exc:
        end_error = traceback.format_exc()
        print("END run failed:", type(exc).__name__, exc)
end_seconds = round(time.time() - start, 1)

SUMMARY["end"] = {
    "seconds": end_seconds,
    "error": end_error,
    "calls": end_log.totals(),
    "gpu": end_gpu.report(),
    "max_prompt_tokens": max((e.get("input_tokens") or 0 for e in end_log.entries), default=0),
    "calls_near_context_limit": sum(bool(e.get("context_near_limit")) for e in end_log.entries),
}
if end_result is not None:
    SUMMARY["end"].update({
        "termination": end_result.termination_reason.value,
        "iterations": end_result.iterations,
        "quality_gate": end_result.quality_gate,
        "acceptance_label": end_result.acceptance_label,
        "trajectory_id": end_result.trajectory_id,
        "trajectory_path": end_result.trajectory_path,
    })
    (RUN_DIR / "answer_end.md").write_text(end_result.answer or "")
with open(RUN_DIR / "call_log_end.jsonl", "w") as f:
    for e in end_log.entries:
        f.write(json.dumps(e) + "\n")
print(f"Done in {end_seconds} s, {end_log.totals()['calls']} model calls.")

## 7. DIRECT baseline (optional, one model call)

In [ ]:
from end_engine.models.base import ModelRequest

if RUN_DIRECT_BASELINE:
    direct_log = CallLog()
    direct_model = OllamaModel(settings, identity, direct_log)
    start = time.time()
    try:
        direct_answer = direct_model.generate(ModelRequest("direct_answer", TASK, {}, None)).content
        direct_error = None
    except Exception as exc:
        direct_answer, direct_error = "", traceback.format_exc()
        print("DIRECT run failed:", exc)
    SUMMARY["direct"] = {"seconds": round(time.time() - start, 1), "error": direct_error, "calls": direct_log.totals()}
    (RUN_DIR / "answer_direct.md").write_text(direct_answer or "")
    print(f"DIRECT done in {SUMMARY['direct']['seconds']} s.")
else:
    SUMMARY["direct"] = "skipped"

## 8. Results

In [ ]:
def show(title, text, limit=4000):
    print("=" * 80); print(title); print("=" * 80)
    print(text[:limit] + ("\n... [truncated; full text in the results zip]" if len(text) > limit else ""))
    print()

e = SUMMARY["end"]
print("GPU:", SUMMARY.get("gpu"), "| Ollama", SUMMARY.get("ollama_version"), "| commit", SUMMARY.get("commit", "")[:10])
print(f"END: {e.get('termination')} after {e.get('iterations')} iteration(s), {e['seconds']} s, "
      f"{e['calls']['calls']} calls, tokens={e['calls']['total_tokens']}, failed attempts={e['calls']['failed_attempts']}")
print("     quality gate:", e.get("quality_gate"), "| acceptance:", e.get("acceptance_label"), "| GPU:", e.get("gpu"))
print(f"     largest prompt: {e['max_prompt_tokens']} of {NUM_CTX} context tokens")
if e["calls_near_context_limit"]:
    print(f"WARNING: {e['calls_near_context_limit']} call(s) used 95%+ of the context window; Ollama may have cut "
          f"the start of those prompts. Raise NUM_CTX in section 1 and rerun.")
if isinstance(SUMMARY["direct"], dict):
    d = SUMMARY["direct"]
    print(f"DIRECT: {d['seconds']} s, tokens={d['calls']['total_tokens']}")
if e.get("error"):
    show("END ERROR", e["error"])
print()
if (RUN_DIR / "answer_end.md").exists():
    show("END ANSWER", (RUN_DIR / "answer_end.md").read_text())
if (RUN_DIR / "answer_direct.md").exists():
    show("DIRECT ANSWER", (RUN_DIR / "answer_direct.md").read_text())

## 9. Save the results

In [ ]:
SUMMARY["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
sh("nvidia-smi > " + str(RUN_DIR / "nvidia_smi_end.txt"), check=False)
(RUN_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=str))
zip_path = shutil.make_archive(str(WORK / f"end_colab_run_{RUN_ID}"), "zip", RUN_DIR)
print("Zip:", zip_path)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = pathlib.Path("/content/drive/MyDrive") / DRIVE_FOLDER
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, dest)
    print("Copied to Google Drive:", dest / pathlib.Path(zip_path).name)
else:
    try:
        from google.colab import files
        files.download(zip_path)
    except ImportError:
        print("Not running in Colab; zip left at", zip_path)